### Import code from a file
To keep workflow components maintainable, reliable, and reusable, each component should contain only the code it needs and explicitly declare its dependencies.

If you want to use your own models in NaaVRE workflows, the preferred approach is to package them as a Python or R package and install the package in a virtual lab. Publishing a package through conda-forge can make it easier to install and share in NaaVRE.

However, turning a model into a package is not always practical. In such cases, the NaaVRE development team may be able to help onboard the model in another way.

During development, you may also want to reuse functions or classes across several workflow components without creating a formal package. You can do this by storing your code in one or more Python or R files and loading those files from cloud storage.

#### Steps

1. **Create one or more code files.**  
   Put reusable code in files with a `.py` or `.r` extension.

2. **Upload the files to cloud storage.**  
   Save the files in [cloud storage](https://naavre.net/docs/NaaVRE_documentation/read-write-files/#Cloud-storage).

   - If you are unsure where to store them, use your personal `naa-vre-user-data` storage.
   - To share files with other users, request access to a virtual-lab bucket or write access to a public bucket.  

3. **Configure the folder parameter.**  
   Set `param_cloud_storage_folder` to the cloud-storage folder containing the code files.

4. **Configure the file-name parameters.**  
   Set `param_first_file_to_import` to the name of the first file to load.

   - To load additional files, use `param_second_file_to_import` and add further parameters if needed.
   - Load files in dependency order. For example, if `timedelta_utilities.py` uses a function defined in `datetime_utilities.py`, load `datetime_utilities.py` first.  

5. **Use the imported code in the component.**  
   Replace the `# Custom code` placeholder with code that imports or calls functions, classes, or other objects from the loaded files.

6. **Test the notebook.**  
   Run the notebook to verify that the files are found, loaded in the correct order, and that the component executes successfully.

In [ ]:
# parameters and configurations
param_datetime = "2026-12-31 23:59:59"

# Import parameters
param_cloud_storage_folder = 'naa-vre-public/training-materials' # could also be set to 'naa-vre-user-data'
param_first_file_to_import = 'datetime_utilities.py'
param_second_file_to_import = 'timedelta_utilities.py'

conf_cloud_storage = '/home/jovyan/Cloud Storage'

In [ ]:
# Workflow component using modules from files

# Boilerplate code to import modules from files
#########################################################################################
import importlib
from importlib import util as importlib_util
import sys
import pathlib

def import_module_from_file(file_folder: str, filename: str):
    filepath = pathlib.Path(file_folder) / filename
    module_name = filename.rsplit('.', 1)[0]
    spec = importlib_util.spec_from_file_location(module_name, filepath)
    module = importlib_util.module_from_spec(spec)
    sys.modules[module_name] = module
    spec.loader.exec_module(module)
    print(f"imported {module_name} from {filepath}")
    return importlib.import_module(module_name)

cloud_storage_path = pathlib.Path(conf_cloud_storage) / param_cloud_storage_folder
#########################################################################################

# Custom code
datetime_utilities = import_module_from_file(cloud_storage_path, param_first_file_to_import)
timedelta_utilities = import_module_from_file(cloud_storage_path, param_second_file_to_import)

timedelta_utilities.print_time_difference(param_datetime)